# Week 01 — Baseline Experiment

**Course:** 人工智慧系統與產業應用（日間部）  
**Theme:** AI System ≠ Model｜Evidence-Driven AI Development

本 Notebook 的目的不是追求最高模型分數，而是完成第一週的最小可重現實驗流程：

**Dataset → Train/Test Split → Baseline → Metric → MLflow → Git Evidence**

---

## Week 1 Check-off

完成本 Notebook 後，學生應能：

- 找到 Dataset 與版本說明
- 說明 Baseline 是什麼
- 執行一個可重現的分類實驗
- 取得至少一個 Metric
- 在 MLflow 留下一次 run、parameter 與 metric
- 說明 Dataset、Baseline、Metric 分別存在哪裡


## 1. AI Opportunity / Problem Definition

### Business / System Problem
我們希望辨識哪些客戶可能流失，以便後續安排人工關懷或留存措施。

### Decision Point
**是否需要將某位客戶標記為「可能流失」並進一步人工檢視？**

### Input
- `age`
- `income`
- `usage_time`
- `complaints`

### Output
- `churn = 0`：未流失
- `churn = 1`：流失

### Baseline
本週使用 **Logistic Regression** 作為簡單分類 Baseline。

### Primary Metric
本週主要教學指標使用 **Accuracy**。  
同時保留 Precision、Recall、F1 與 Confusion Matrix，讓學生知道單一 Accuracy 並不足以描述所有失敗情境。

### Failure Path
若模型判斷錯誤，不直接自動執行高風險決策；本案例應由人工進一步檢視。  
此資料為教學用合成資料，不可據此做真實商業決策。


## 2. Import Packages

若出現套件不存在，可先在終端機執行：

```bash
pip install -r requirements.txt
```


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("mlflow:", mlflow.__version__)


## 3. Resolve Repository Paths

此 Cell 會自動判斷你是從 repo 根目錄啟動 Jupyter，或直接在 `notebooks/` 內啟動。

預期 repo 結構：

```text
ai-system-course/
├── data/
│   ├── README.md
│   └── week01_customer_churn_sample.csv
├── notebooks/
│   └── week01_baseline.ipynb
├── mlruns/
└── requirements.txt
```


In [ ]:
cwd = Path.cwd().resolve()

if (cwd / "data").exists():
    repo_root = cwd
elif (cwd.parent / "data").exists():
    repo_root = cwd.parent
else:
    raise FileNotFoundError(
        "找不到 data/ 資料夾。請從課程 repo 根目錄或 notebooks/ 目錄執行 Notebook。"
    )

data_path = repo_root / "data" / "week01_customer_churn_sample.csv"
data_readme_path = repo_root / "data" / "README.md"
mlruns_path = repo_root / "mlruns"
artifacts_path = repo_root / "artifacts"

artifacts_path.mkdir(exist_ok=True)

print("Repository root:", repo_root)
print("Dataset path:", data_path)
print("Dataset README:", data_readme_path)
print("MLflow directory:", mlruns_path)


## 4. Load Dataset

資料集版本資訊應記錄在：

`data/README.md`

本課程第一週要求學生能回答：

> Dataset 版本在哪裡？


In [ ]:
if not data_path.exists():
    raise FileNotFoundError(f"找不到 Dataset: {data_path}")

df = pd.read_csv(data_path)

print("Shape:", df.shape)
display(df.head())


## 5. Inspect Dataset

先看資料，不要直接訓練模型。

至少確認：

- 欄位名稱
- 缺失值
- Target 分布
- 是否出現明顯異常


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nTarget distribution:")
print(df["churn"].value_counts().sort_index())

print("\nSummary:")
display(df.describe())


## 6. Define Features and Target

`X`：模型輸入  
`y`：模型要預測的 Target


In [ ]:
FEATURES = ["age", "income", "usage_time", "complaints"]
TARGET = "churn"

X = df[FEATURES]
y = df[TARGET]

print("Features:", FEATURES)
print("Target:", TARGET)


## 7. Train / Test Split

本週固定使用：

- `test_size = 0.20`
- `random_state = 42`
- `stratify = y`

固定亂數種子是為了讓全班先得到可重現結果。  
這不表示這組設定是最佳設定。


In [ ]:
TEST_SIZE = 0.20
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("\nTrain target distribution:")
print(y_train.value_counts().sort_index())
print("\nTest target distribution:")
print(y_test.value_counts().sort_index())


## 8. Build Baseline Model

使用：

**StandardScaler → LogisticRegression**

使用 Pipeline 的目的，是把前處理與模型放在同一個可重現流程中。


In [ ]:
MODEL_NAME = "LogisticRegression"
MAX_ITER = 1000

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=MAX_ITER,
        random_state=RANDOM_STATE
    )),
])

model.fit(X_train, y_train)

print("Baseline trained:", MODEL_NAME)


## 9. Evaluate Baseline

第一週至少需要一個 Metric；這裡同時保留：

- Accuracy
- Precision
- Recall
- F1


In [ ]:
pred = model.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall": recall_score(y_test, pred, zero_division=0),
    "f1": f1_score(y_test, pred, zero_division=0),
}

for name, value in metrics.items():
    print(f"{name}: {value:.4f}")


## 10. Confusion Matrix

Confusion Matrix 可以幫助我們回答：

> 模型「錯在哪裡」？

這比只看 Accuracy 更接近 Evidence-Driven AI System Development 的要求。


In [ ]:
cm = confusion_matrix(y_test, pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn (0)", "Churn (1)"]
)

disp.plot()
plt.title("Week 01 Baseline — Confusion Matrix")
plt.tight_layout()

cm_path = artifacts_path / "week01_confusion_matrix.png"
plt.savefig(cm_path, dpi=150)
plt.show()

print("Saved artifact:", cm_path)


## 11. Inspect Error Cases

不要只保存最好結果。  
把預測錯誤的資料列留下來，才能進一步做 Failure Analysis。


In [ ]:
result_df = X_test.copy()
result_df["actual"] = y_test.values
result_df["predicted"] = pred

errors_df = result_df[result_df["actual"] != result_df["predicted"]]

print("Number of error cases:", len(errors_df))
display(errors_df)


## 12. Log Experiment to MLflow

本週最低要求：

- 1 個 run
- 至少 1 個 parameter
- 至少 1 個 metric

這裡會把完整 Baseline 設定與主要 metrics 都記錄下來。

> 注意：本 Notebook 使用 repo 內的 `mlruns/` 作為本機 MLflow Tracking Store。


In [ ]:
mlruns_uri = mlruns_path.resolve().as_uri()
mlflow.set_tracking_uri(mlruns_uri)
mlflow.set_experiment("week01-baseline")

with mlflow.start_run(run_name="logistic-regression-baseline") as run:
    mlflow.log_param("dataset_version", "v1.0")
    mlflow.log_param("model", MODEL_NAME)
    mlflow.log_param("test_size", TEST_SIZE)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("max_iter", MAX_ITER)
    mlflow.log_param("features", ",".join(FEATURES))

    for name, value in metrics.items():
        mlflow.log_metric(name, float(value))

    if cm_path.exists():
        mlflow.log_artifact(str(cm_path), artifact_path="evaluation")

    run_id = run.info.run_id

print("MLflow run completed.")
print("Run ID:", run_id)
print("Tracking URI:", mlflow.get_tracking_uri())


## 13. Save a Small Experiment Summary

除了 MLflow，也將本次實驗摘要輸出成 JSON，方便學生理解「Evidence 在哪裡」。


In [ ]:
summary = {
    "dataset": "week01_customer_churn_sample.csv",
    "dataset_version": "v1.0",
    "model": MODEL_NAME,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "metrics": {k: float(v) for k, v in metrics.items()},
    "mlflow_run_id": run_id,
}

summary_path = artifacts_path / "week01_experiment_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved:", summary_path)
print(json.dumps(summary, ensure_ascii=False, indent=2))


## 14. Evidence Check

完成後，你應能回答：

### Dataset 在哪裡？
`data/week01_customer_churn_sample.csv`

### Dataset version 在哪裡？
`data/README.md` → v1.0

### Baseline 在哪裡？
本 Notebook 的 **Build Baseline Model** 區段  
模型：`LogisticRegression`

### Metric 在哪裡？
1. 本 Notebook 的 **Evaluate Baseline**
2. MLflow run
3. `artifacts/week01_experiment_summary.json`

### Error / Failure evidence 在哪裡？
1. 本 Notebook 的 **Inspect Error Cases**
2. `artifacts/week01_confusion_matrix.png`

### Experiment version 在哪裡？
Git / Gitea commit history


## 15. Student Challenge

不要修改 Dataset。

只修改以下其中一項，再重新執行實驗：

- `TEST_SIZE`
- Logistic Regression 的參數
- Feature 組合

然後回答：

1. Metric 有沒有改變？
2. 為什麼可能改變？
3. 新結果是否真的比 Baseline 更可信？
4. 你在 MLflow 中能不能找到兩次 run 的差異？

完成後再進行 Git commit / push。


## 16. Git Evidence

在終端機確認：

```bash
git status
git add .
git commit -m "Complete Week 1 baseline experiment"
git push
```

若只想查看尚未 commit 的修改：

```bash
git diff
```

---

## Week 1 Exit Check

- [ ] Notebook 可從頭執行
- [ ] Dataset v1.0 可找到
- [ ] Baseline 可說明
- [ ] Metric 可找到
- [ ] Error cases 可找到
- [ ] MLflow 至少有 1 run
- [ ] Git commit / push 完成
- [ ] 能用一句話說明：**AI Model Works ≠ AI System Works**
